# Weight sensitivity


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp").exists():
            return candidate
    for candidate in [cwd, *cwd.parents]:
        nested = candidate / "analysis"
        if (nested / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp").exists():
            return nested
    raise FileNotFoundError("Cannot locate analysis project root.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
TABLE_DIR = PROJECT_ROOT / "supplementary" / "tables"
FIG_DIR = PROJECT_ROOT / "supplementary" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

BASE_HEAT_COL = "HE_perpop"
BASE_FLOOD_COL = "sl_sevexp"
EDI_COL = "EDI_qmean"
COUNTRY_COL = "iso3"
POP_COL = "sl_pop"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

raw_gdf = gpd.read_file(DATA_PATH)
raw_gdf[COUNTRY_COL] = raw_gdf[COUNTRY_COL].astype(str)


def zscore(x):
    x = pd.to_numeric(x, errors="coerce")
    std = x.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return x * np.nan
    return (x - x.mean()) / std


def prepare_ratio(numerator_col, denominator_col, conversion, heat_col=BASE_HEAT_COL, flood_col=BASE_FLOOD_COL, threshold_q=0.75):
    d = raw_gdf.copy()
    d["ratio_numer"] = pd.to_numeric(d[numerator_col], errors="coerce")
    d["ratio_denom"] = pd.to_numeric(d[denominator_col], errors="coerce")
    d["ratio_weight"] = d["ratio_denom"]
    d["conversion_ratio"] = d["ratio_numer"] / d["ratio_denom"]
    invalid = (d["ratio_denom"] <= 0) | (d["ratio_numer"] < 0) | (d["conversion_ratio"] < 0)
    d.loc[invalid, ["conversion_ratio", "ratio_weight"]] = np.nan
    d["log_ratio"] = np.log1p(d["conversion_ratio"])
    d["heat_z"] = zscore(d[heat_col])
    d["flood_z"] = zscore(d[flood_col])
    d["edi_z"] = zscore(d[EDI_COL])
    d["logpop_z"] = zscore(np.log1p(d["ratio_weight"]))
    d["slpop_weight"] = pd.to_numeric(d[POP_COL], errors="coerce")
    d["high_heat"] = (pd.to_numeric(d[heat_col], errors="coerce") >= pd.to_numeric(d[heat_col], errors="coerce").quantile(threshold_q)).astype(int)
    d["high_flood"] = (pd.to_numeric(d[flood_col], errors="coerce") >= pd.to_numeric(d[flood_col], errors="coerce").quantile(threshold_q)).astype(int)
    d["joint_heat_flood"] = ((d["high_heat"] == 1) & (d["high_flood"] == 1)).astype(int)
    d["conversion"] = conversion
    needed = ["log_ratio", "conversion_ratio", "ratio_weight", "heat_z", "flood_z", "edi_z", "logpop_z", COUNTRY_COL]
    return d.dropna(subset=needed).copy()


def conversion_datasets(heat_col=BASE_HEAT_COL, flood_col=BASE_FLOOD_COL, threshold_q=0.75):
    return {
        "infection_to_incidence_ratio": prepare_ratio("sl_pfinc", "sl_pfinf", "infection_to_incidence_ratio", heat_col, flood_col, threshold_q),
        "incidence_to_mortality_ratio": prepare_ratio("sl_pfmort", "sl_pfinc", "incidence_to_mortality_ratio", heat_col, flood_col, threshold_q),
    }


def fit_clustered(formula, data, weight_col=None):
    model_df = data.copy()
    if weight_col is None:
        fit = smf.ols(formula, data=model_df).fit(cov_type="cluster", cov_kwds={"groups": model_df[COUNTRY_COL]})
    else:
        model_df = model_df.dropna(subset=[weight_col]).copy()
        model_df = model_df[pd.to_numeric(model_df[weight_col], errors="coerce") > 0].copy()
        fit = smf.wls(formula, data=model_df, weights=model_df[weight_col]).fit(
            cov_type="cluster", cov_kwds={"groups": model_df[COUNTRY_COL]}
        )
    return fit, model_df


def collect_terms(model, rows, analysis, conversion, variant, formula, terms, nobs):
    for term in terms:
        if term not in model.params.index:
            continue
        coef = float(model.params[term])
        se = float(model.bse[term])
        rows.append({
            "analysis": analysis,
            "conversion": conversion,
            "variant": variant,
            "formula": formula,
            "term": term,
            "n": int(nobs),
            "coef": coef,
            "se": se,
            "ci_low": coef - 1.96 * se,
            "ci_high": coef + 1.96 * se,
            "pval": float(model.pvalues[term]),
            "effect_pct": 100 * (np.exp(coef) - 1),
            "ci_low_pct": 100 * (np.exp(coef - 1.96 * se) - 1),
            "ci_high_pct": 100 * (np.exp(coef + 1.96 * se) - 1),
        })


TERM_LABELS = {
    "edi_z": "EDI (+1 s.d.)",
    "heat_z": "Heat (+1 s.d.)",
    "flood_z": "Flood (+1 s.d.)",
    "heat_z:flood_z": "Heat x flood",
    "high_heat": "High heat",
    "high_flood": "High flood",
    "joint_heat_flood": "Joint high\nheat + flood",
}


def save_forest_plot(df, out_base, title, variant_col="variant", max_variants=None, baseline_variant=None, legend_y=0.1, legend_fontsize=8.5):
    d = df.copy()
    if max_variants is not None:
        keep = list(d[variant_col].drop_duplicates())[:max_variants]
        d = d[d[variant_col].isin(keep)].copy()
    conversions = list(d["conversion"].drop_duplicates())
    all_variants = list(d[variant_col].drop_duplicates())
    fig, axes = plt.subplots(1, len(conversions), figsize=(10.2, 5), squeeze=False)
    colors = plt.cm.tab10(np.linspace(0, 1, max(3, len(all_variants))))
    color_map = dict(zip(all_variants, colors))
    axes_flat = axes.ravel()
    for panel_idx, (ax, conversion) in enumerate(zip(axes_flat, conversions)):
        sub = d[d["conversion"].eq(conversion)].copy()
        terms = list(sub["term"].drop_duplicates())
        y_base = np.arange(len(terms))
        variants = list(sub[variant_col].drop_duplicates())
        offsets = np.linspace(-0.30, 0.30, len(variants)) if len(variants) > 1 else [0]
        for offset, variant in zip(offsets, variants):
            s = sub[sub[variant_col].eq(variant)].copy()
            ypos = [terms.index(t) + offset for t in s["term"]]
            x = s["effect_pct"].to_numpy()
            lo = s["ci_low_pct"].to_numpy()
            hi = s["ci_high_pct"].to_numpy()
            label = f"{variant} (original)" if variant == baseline_variant else str(variant)
            ax.errorbar(
                x, ypos, xerr=[x - lo, hi - x], fmt="o", ms=4,
                capsize=2.5, lw=0.9, color=color_map[variant], label=label, alpha=0.95,
            )
            if variant == baseline_variant:
                ax.scatter(x, ypos, s=46, facecolors="none", edgecolors="black", linewidths=0.9, zorder=5)
        ax.axvline(0, color="0.35", lw=0.8)
        ax.set_yticks(y_base)
        ax.set_yticklabels([TERM_LABELS.get(term, term) for term in terms])
        ax.set_title(conversion.replace("_", " "))
        ax.set_xlabel("Effect on log1p conversion ratio (%)")
        ax.grid(axis="x", color="0.90", lw=0.6)
    handles, labels = axes_flat[-1].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        frameon=False,
        ncol=2 if len(labels) > 3 else len(labels),
        loc="lower center",
        bbox_to_anchor=(0.5, legend_y),
        borderaxespad=0,
        handlelength=1.4,
        handletextpad=0.45,
        labelspacing=0.35,
        fontsize=legend_fontsize,
    )
    fig.suptitle(title, x=0.01, y=0.965, ha="left", fontweight="bold")
    # if baseline_variant is not None:
    #     fig.text(0.01, 0.925, "Black open circles = original main-text estimates", ha="left", va="top", fontsize=7.5)
    fig.tight_layout(rect=[0, 0.17, 1, 0.94], w_pad=2.0)
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.show()


def save_weight_profile_plot(df, out_base, title, baseline_variant="Denominator WLS"):
    d = df.copy()
    conversion_order = ["infection_to_incidence_ratio", "incidence_to_mortality_ratio"]
    conversions = [c for c in conversion_order if c in set(d["conversion"])]
    variant_order = [
        "Unweighted OLS",
        "Denominator WLS",
        "Sqrt-denominator WLS",
        "Capped-denominator WLS",
    ]
    variant_order = [v for v in variant_order if v in set(d["variant"])]
    term_order = ["heat_z", "flood_z", "heat_z:flood_z", "edi_z"]
    term_order = [t for t in term_order if t in set(d["term"])]
    term_colors = {
        "heat_z": "#C44E52",
        "flood_z": "#4C72B0",
        "heat_z:flood_z": "#55A868",
        "edi_z": "#8172B2",
    }
    x = np.arange(len(variant_order))
    baseline_x = variant_order.index(baseline_variant) if baseline_variant in variant_order else None
    fig, axes = plt.subplots(1, len(conversions), figsize=(8.6, 4.4), squeeze=False, sharex=True)
    axes_flat = axes.ravel()

    for ax, conversion in zip(axes_flat, conversions):
        sub = d[d["conversion"].eq(conversion)].copy()
        for term in term_order:
            s = sub[sub["term"].eq(term)].copy()
            s["variant_order"] = s["variant"].map({v: i for i, v in enumerate(variant_order)})
            s = s.dropna(subset=["variant_order"]).sort_values("variant_order")
            xpos = s["variant_order"].to_numpy(dtype=float)
            y = s["effect_pct"].to_numpy()
            lo = s["ci_low_pct"].to_numpy()
            hi = s["ci_high_pct"].to_numpy()
            color = term_colors.get(term, "0.35")
            ax.errorbar(
                xpos,
                y,
                yerr=[y - lo, hi - y],
                fmt="o-",
                ms=4,
                lw=1.1,
                capsize=2.2,
                color=color,
                ecolor=color,
                label=TERM_LABELS.get(term, term),
                alpha=0.95,
            )
            if baseline_x is not None:
                base = s[s["variant"].eq(baseline_variant)]
                if not base.empty:
                    ax.scatter(
                        [baseline_x],
                        base["effect_pct"],
                        s=48,
                        facecolors="none",
                        edgecolors="black",
                        linewidths=0.9,
                        zorder=5,
                    )
        if baseline_x is not None:
            ax.axvline(baseline_x, color="0.25", lw=0.7, ls=":")
        ax.axhline(0, color="0.35", lw=0.8)
        ax.set_title(conversion.replace("_", " "))
        ax.set_xticks(x)
        ax.set_xticklabels([v.replace(" WLS", "\nWLS").replace(" OLS", "\nOLS") for v in variant_order])
        ax.set_xlabel("Weighting specification")
        ax.grid(axis="y", color="0.90", lw=0.6)
    axes_flat[0].set_ylabel("Effect on log1p conversion ratio (%)")
    handles, labels = axes_flat[-1].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        frameon=False,
        ncol=len(labels),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.06),
        borderaxespad=0,
        handlelength=1.7,
        handletextpad=0.45,
        fontsize=8.5,
    )
    fig.suptitle(title, x=0.01, y=0.99, ha="left", fontweight="bold")
    fig.tight_layout(rect=[0, 0.15, 1, 0.94], w_pad=2.0)
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.show()


# Weight sensitivity: compare how much the direct conversion-rate coefficients depend on weighting.
base_data = conversion_datasets()
continuous_formula = "log_ratio ~ heat_z + flood_z + heat_z:flood_z + edi_z + logpop_z"
joint_formula = "log_ratio ~ high_heat + high_flood + joint_heat_flood + edi_z + logpop_z"
weight_rows = []

for conversion, source_df in base_data.items():
    d = source_df.copy()
    p99 = float(d["ratio_weight"].quantile(0.99))
    d["weight_denominator"] = d["ratio_weight"]
    d["weight_sqrt_denominator"] = np.sqrt(d["ratio_weight"])
    d["weight_capped_denominator_p99"] = d["ratio_weight"].clip(upper=p99)
    weight_specs = [
        ("Unweighted OLS", None),
        ("Denominator WLS", "weight_denominator"),
        ("Sqrt-denominator WLS", "weight_sqrt_denominator"),
        ("Capped-denominator WLS", "weight_capped_denominator_p99"),
    ]
    for variant, weight_col in weight_specs:
        for analysis, formula, terms in [
            ("Continuous exposure model", continuous_formula, ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"]),
            ("High exposure model", joint_formula, ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"]),
        ]:
            model, model_df = fit_clustered(formula, d, weight_col)
            collect_terms(model, weight_rows, analysis, conversion, variant, formula, terms, len(model_df))

weight_sensitivity = pd.DataFrame(weight_rows)
weight_sensitivity.to_csv(TABLE_DIR / "sensitivity_weight_models.csv", index=False, encoding="utf-8-sig")

weight_plot = weight_sensitivity[
    weight_sensitivity["analysis"].eq("Continuous exposure model")
    & weight_sensitivity["term"].isin(["heat_z", "flood_z", "heat_z:flood_z", "edi_z"])
].copy()
save_forest_plot(
    weight_plot,
    FIG_DIR / "sensitivity_weight_continuous_model",
    "Weight sensitivity: continuous exposure model",
    baseline_variant="Denominator WLS",
)


# Exposure-definition sensitivity


In [ ]:
# Exposure-definition sensitivity: refit the continuous model with alternative heat and flood metrics.
# Baseline model uses HE_perpop and sl_sevexp, matching Figures 2-4.
heat_candidates = [
    ("HE_perpop", "Heat: SPW WBGT days"),
    ("Heat_days_", "Heat: ADM mean WBGT days"),
    ("HI_day", "Heat: HI days"),
]
flood_candidates = [
    ("sl_sevexp", "Flood: SPW severity"),
    ("fl_sevexp", "Flood: area severity"),
    ("sl_flpct", "Flood: exposed slum-pop share"),
    ("fl_pct", "Flood: flooded area share"),
    ("fl_maxsev", "Flood: maximum severity"),
]

available_heat = [(col, label) for col, label in heat_candidates if col in raw_gdf.columns]
available_flood = [(col, label) for col, label in flood_candidates if col in raw_gdf.columns]

exposure_rows = []
for heat_col, heat_label in available_heat:
    for flood_col, flood_label in available_flood:
        variant = f"{heat_col} + {flood_col}"
        data_by_conversion = conversion_datasets(heat_col=heat_col, flood_col=flood_col)
        for conversion, source_df in data_by_conversion.items():
            model, model_df = fit_clustered(continuous_formula, source_df, "ratio_weight")
            collect_terms(
                model,
                exposure_rows,
                "Continuous exposure-definition model",
                conversion,
                variant,
                continuous_formula,
                ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"],
                len(model_df),
            )
            for row in exposure_rows[-5:]:
                row["heat_col"] = heat_col
                row["heat_definition"] = heat_label
                row["flood_col"] = flood_col
                row["flood_definition"] = flood_label

exposure_sensitivity = pd.DataFrame(exposure_rows)
exposure_sensitivity.to_csv(TABLE_DIR / "sensitivity_exposure_definition_models.csv", index=False, encoding="utf-8-sig")

# Main exposure terms only, with a compact heat/flood combination label.
exposure_plot = exposure_sensitivity[exposure_sensitivity["term"].isin(["heat_z", "flood_z", "heat_z:flood_z"])] .copy()
exposure_plot["definition"] = exposure_plot["heat_col"] + " | " + exposure_plot["flood_col"]

# Baseline-vs-alternative summary for quick reading.
baseline_variant = "HE_perpop + sl_sevexp"
baseline = exposure_plot[exposure_plot["variant"].eq(baseline_variant)][
    ["conversion", "term", "effect_pct"]
].rename(columns={"effect_pct": "baseline_effect_pct"})
exposure_delta = exposure_plot.merge(baseline, on=["conversion", "term"], how="left")
exposure_delta["effect_pct_minus_baseline"] = exposure_delta["effect_pct"] - exposure_delta["baseline_effect_pct"]
exposure_delta["ci_low_pct_minus_baseline"] = exposure_delta["ci_low_pct"] - exposure_delta["baseline_effect_pct"]
exposure_delta["ci_high_pct_minus_baseline"] = exposure_delta["ci_high_pct"] - exposure_delta["baseline_effect_pct"]
exposure_delta.to_csv(TABLE_DIR / "sensitivity_exposure_definition_delta_from_baseline.csv", index=False, encoding="utf-8-sig")


def plot_exposure_delta_forest(df, out_base, baseline_variant="HE_perpop + sl_sevexp"):
    terms = ["heat_z", "flood_z", "heat_z:flood_z"]
    term_labels = {
        "heat_z": "Heat exposure",
        "flood_z": "Flood exposure",
        "heat_z:flood_z": "Heat x flood",
    }
    conversions = list(df["conversion"].drop_duplicates())
    alt = df[~df["variant"].eq(baseline_variant)].copy()
    alt = alt.dropna(subset=["effect_pct_minus_baseline"])
    summary = (
        alt.groupby(["conversion", "term"], as_index=False)
        .agg(
            n_definitions=("variant", "nunique"),
            min_delta=("effect_pct_minus_baseline", "min"),
            q25_delta=("effect_pct_minus_baseline", lambda x: x.quantile(0.25)),
            median_delta=("effect_pct_minus_baseline", "median"),
            q75_delta=("effect_pct_minus_baseline", lambda x: x.quantile(0.75)),
            max_delta=("effect_pct_minus_baseline", "max"),
        )
    )
    summary.to_csv(TABLE_DIR / "sensitivity_exposure_definition_delta_summary.csv", index=False, encoding="utf-8-sig")

    fig, axes = plt.subplots(1, len(conversions), figsize=(8.2, 4), sharey=True, squeeze=False)
    axes = axes.ravel()
    y_lookup = {term: len(terms) - 1 - idx for idx, term in enumerate(terms)}
    range_color = "#8FBAD9"
    iqr_color = "#2C7FB8"
    median_color = "#1F78B4"

    for ax, conversion in zip(axes, conversions):
        sub = summary[summary["conversion"].eq(conversion)].copy()
        sub["y"] = sub["term"].map(y_lookup)
        for _, row in sub.iterrows():
            y = row["y"]
            ax.hlines(y, row["min_delta"], row["max_delta"], color=range_color, lw=1.1, zorder=1)
            ax.hlines(y, row["q25_delta"], row["q75_delta"], color=iqr_color, lw=4.0, zorder=2)
            ax.scatter(row["median_delta"], y, s=28, color=median_color, zorder=3)
            ax.scatter(0, y, s=48, facecolors="none", edgecolors="black", linewidths=1.0, zorder=4)
        x_min = sub["min_delta"].min()
        x_max = sub["max_delta"].max()
        max_abs = max(abs(x_min), abs(x_max), 1e-6) * 1.18
        ax.set_xlim(-max_abs, max_abs)
        ax.axvline(0, color="0.25", lw=0.8)
        ax.grid(axis="x", color="0.90", lw=0.6)
        ax.set_title(conversion.replace("_", " "))
        ax.set_xlabel("Difference from original effect (percentage points)")
        ax.set_ylim(-0.55, len(terms) - 0.45)

    axes[0].set_yticks([y_lookup[t] for t in terms])
    axes[0].set_yticklabels([term_labels[t] for t in terms])
    for ax in axes[1:]:
        ax.tick_params(axis="y", labelleft=False)

    handles = [
        mpl.lines.Line2D([0], [0], color=range_color, lw=1.2, label="Min-max across alternatives"),
        mpl.lines.Line2D([0], [0], color=iqr_color, lw=4.0, label="IQR across alternatives"),
        mpl.lines.Line2D([0], [0], color=median_color, marker="o", lw=0, ms=4.5, label="Median alternative"),
        mpl.lines.Line2D([0], [0], color="none", marker="o", markerfacecolor="none", markeredgecolor="black", ms=5.5, label="Original main-text definition"),
    ]
    fig.legend(
        handles=handles,
        frameon=False,
        ncol=4,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.1),
        borderaxespad=0,
        handlelength=1.8,
        columnspacing=1.2,
        handletextpad=0.45,
        fontsize=8.5
    )
    fig.suptitle("Exposure-definition sensitivity: difference from original", x=0.01, y=0.965, ha="left", fontweight="bold")
    fig.tight_layout(rect=[0, 0.18, 1, 0.95], w_pad=2.2)
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.show()


plot_exposure_delta_forest(exposure_delta, FIG_DIR / "sensitivity_exposure_definition_delta_forest")


# High-exposure threshold sensitivity


In [ ]:
# High-exposure threshold sensitivity: refit the high heat/flood model at alternative quantile cutoffs.
def save_threshold_trend_plot(df, out_base, title, baseline_variant="q75"):
    d = df.copy()
    conversion_order = ["infection_to_incidence_ratio", "incidence_to_mortality_ratio"]
    conversions = [c for c in conversion_order if c in set(d["conversion"])]
    term_order = ["high_heat", "high_flood", "joint_heat_flood"]
    term_order = [t for t in term_order if t in set(d["term"])]
    term_colors = {
        "high_heat": "#C44E52",
        "high_flood": "#4C72B0",
        "joint_heat_flood": "#55A868",
    }
    fig, axes = plt.subplots(1, len(conversions), figsize=(8.2, 4.2), squeeze=False, sharex=True)
    axes_flat = axes.ravel()
    for ax, conversion in zip(axes_flat, conversions):
        sub = d[d["conversion"].eq(conversion)].copy()
        for term in term_order:
            s = sub[sub["term"].eq(term)].copy()
            if "threshold_quantile" not in s:
                s["threshold_quantile"] = pd.to_numeric(s["variant"].str.replace("q", ""), errors="coerce") / 100
            s = s.sort_values("threshold_quantile")
            x = (s["threshold_quantile"].to_numpy(dtype=float) * 100)
            y = s["effect_pct"].to_numpy()
            lo = s["ci_low_pct"].to_numpy()
            hi = s["ci_high_pct"].to_numpy()
            color = term_colors.get(term, "0.35")
            ax.errorbar(
                x,
                y,
                yerr=[y - lo, hi - y],
                fmt="o-",
                ms=4.2,
                lw=1.2,
                capsize=2.3,
                color=color,
                ecolor=color,
                label=TERM_LABELS.get(term, term).replace("\n", " "),
                alpha=0.95,
            )
            base = s[s["variant"].eq(baseline_variant)]
            if not base.empty:
                ax.scatter(
                    base["threshold_quantile"] * 100,
                    base["effect_pct"],
                    s=50,
                    facecolors="none",
                    edgecolors="black",
                    linewidths=0.9,
                    zorder=5,
                )
        ax.axhline(0, color="0.35", lw=0.8)
        ax.axvline(75, color="0.25", lw=0.7, ls=":")
        ax.set_title(conversion.replace("_", " "))
        ax.set_xlabel("High-exposure percentile cutoff")
        ax.set_xticks([70, 75, 80])
        ax.set_xticklabels(["70th", "75th", "80th"])
        ax.grid(axis="y", color="0.90", lw=0.6)
    axes_flat[0].set_ylabel("Effect on log1p conversion ratio (%)")
    handles, labels = axes_flat[-1].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        frameon=False,
        ncol=len(labels),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.075),
        borderaxespad=0,
        handlelength=1.7,
        handletextpad=0.45,
        fontsize=8.5,
    )
    fig.suptitle(title, x=0.01, y=0.965, ha="left", fontweight="bold")
    # fig.text(0.01, 0.95, "Black open circles = original main-text estimates", ha="left", va="top", fontsize=7.5)
    fig.tight_layout(rect=[0, 0.17, 1, 0.95], w_pad=2.0)
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.show()

threshold_rows = []
threshold_specs = [0.70, 0.75, 0.80]

for threshold_q in threshold_specs:
    data_by_conversion = conversion_datasets(threshold_q=threshold_q)
    for conversion, source_df in data_by_conversion.items():
        model, model_df = fit_clustered(joint_formula, source_df, "ratio_weight")
        collect_terms(
            model,
            threshold_rows,
            "High exposure threshold model",
            conversion,
            f"q{int(threshold_q * 100)}",
            joint_formula,
            ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"],
            len(model_df),
        )
        threshold_rows[-5]["threshold_quantile"] = threshold_q
        threshold_rows[-4]["threshold_quantile"] = threshold_q
        threshold_rows[-3]["threshold_quantile"] = threshold_q
        threshold_rows[-2]["threshold_quantile"] = threshold_q
        threshold_rows[-1]["threshold_quantile"] = threshold_q

threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_sensitivity.to_csv(TABLE_DIR / "sensitivity_high_exposure_threshold_models.csv", index=False, encoding="utf-8-sig")

threshold_plot = threshold_sensitivity[threshold_sensitivity["term"].isin(["high_heat", "high_flood", "joint_heat_flood"])].copy()
save_threshold_trend_plot(
    threshold_plot,
    FIG_DIR / "sensitivity_high_exposure_threshold_model",
    "High-exposure threshold sensitivity",
    baseline_variant="q75",
)

# Compact significance-direction table for manuscript checking.
threshold_direction = (
    threshold_plot.assign(
        sign=np.where(threshold_plot["coef"] > 0, "+", "-"),
        p_lt_0_05=threshold_plot["pval"] < 0.05,
    )
    [["conversion", "variant", "threshold_quantile", "term", "effect_pct", "ci_low_pct", "ci_high_pct", "pval", "sign", "p_lt_0_05", "n"]]
)
threshold_direction.to_csv(TABLE_DIR / "sensitivity_high_exposure_threshold_direction_summary.csv", index=False, encoding="utf-8-sig")


# Region fixed-effects sensitivity

In [ ]:
# Region fixed-effects sensitivity: compare Figure 2/3 main models with and without broad African-region fixed effects.
REGION_COL = "region_fe"
COUNTRY_REGION_PATH = PROJECT_ROOT / "data" / "country" / "country_SSA_HE2.shp"

country_region = gpd.read_file(COUNTRY_REGION_PATH)[["country_id", "region"]].copy()
country_region["country_id"] = pd.to_numeric(country_region["country_id"], errors="coerce").astype("Int64")
country_region = country_region.dropna(subset=["country_id", "region"]).drop_duplicates("country_id")
region_map = dict(zip(country_region["country_id"].astype(int), country_region["region"].astype(str)))
admin_country_id = (pd.to_numeric(raw_gdf["adm2ID"], errors="coerce") // 10000).astype("Int64")
raw_gdf[REGION_COL] = admin_country_id.map(region_map)

region_model_specs = [
    {
        "analysis": "Continuous exposure model",
        "base_formula": continuous_formula,
        "terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"],
        "plot_terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z"],
    },
    {
        "analysis": "High exposure model",
        "base_formula": joint_formula,
        "terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"],
        "plot_terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z"],
    },
]

region_rows = []
base_data = conversion_datasets()
for conversion, source_df in base_data.items():
    source_df = source_df.dropna(subset=[REGION_COL]).copy()
    for spec in region_model_specs:
        variants = [
            ("Baseline", spec["base_formula"]),
            ("Region fixed effects", f"{spec['base_formula']} + C({REGION_COL})"),
        ]
        for variant, formula in variants:
            model, model_df = fit_clustered(formula, source_df, "ratio_weight")
            before = len(region_rows)
            collect_terms(
                model,
                region_rows,
                spec["analysis"],
                conversion,
                variant,
                formula,
                spec["terms"],
                len(model_df),
            )
            for row in region_rows[before:]:
                row["region_count"] = int(model_df[REGION_COL].nunique())
                row["country_count"] = int(model_df[COUNTRY_COL].nunique())

region_fe_sensitivity = pd.DataFrame(region_rows)
region_fe_sensitivity.to_csv(TABLE_DIR / "sensitivity_region_fixed_effects_models.csv", index=False, encoding="utf-8-sig")

region_fe_plot = region_fe_sensitivity[
    region_fe_sensitivity["term"].isin(["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "high_heat", "high_flood", "joint_heat_flood"])
].copy()


def save_region_fe_scatter_plot(df, out_base, title, model_specs):
    conversion_order = ["infection_to_incidence_ratio", "incidence_to_mortality_ratio"]
    conversions = [c for c in conversion_order if c in set(df["conversion"])]
    term_colors = {
        "heat_z": "#C44E52",
        "flood_z": "#4C72B0",
        "heat_z:flood_z": "#55A868",
        "high_heat": "#C44E52",
        "high_flood": "#4C72B0",
        "joint_heat_flood": "#55A868",
        "edi_z": "#8172B2",
    }
    short_labels = {
        "heat_z": "Heat",
        "flood_z": "Flood",
        "heat_z:flood_z": "Heat x flood",
        "high_heat": "High heat",
        "high_flood": "High flood",
        "joint_heat_flood": "Joint high",
        "edi_z": "EDI",
    }
    fig, axes = plt.subplots(len(model_specs), len(conversions), figsize=(6.6, 5.9), squeeze=False)

    for row_idx, spec in enumerate(model_specs):
        terms = [term for term in spec["plot_terms"] if term in set(df["term"])]
        for col_idx, conversion in enumerate(conversions):
            ax = axes[row_idx, col_idx]
            sub = df[(df["analysis"].eq(spec["analysis"])) & (df["conversion"].eq(conversion)) & (df["term"].isin(terms))].copy()
            wide = sub.pivot_table(
                index="term",
                columns="variant",
                values="effect_pct",
                aggfunc="first",
            ).reindex(terms)
            wide = wide.dropna(subset=["Baseline", "Region fixed effects"])
            if wide.empty:
                ax.axis("off")
                continue

            x = wide["Baseline"].to_numpy(dtype=float)
            y = wide["Region fixed effects"].to_numpy(dtype=float)
            finite = np.r_[x[np.isfinite(x)], y[np.isfinite(y)]]
            lo, hi = np.nanmin(finite), np.nanmax(finite)
            span = hi - lo
            pad = max(span * 0.24, 0.06 if max(abs(lo), abs(hi)) < 1 else 1.0)
            lo, hi = lo - pad, hi + pad
            ax.plot([lo, hi], [lo, hi], color="0.45", lw=0.9, ls="--", zorder=1)
            ax.axhline(0, color="0.86", lw=0.8, zorder=0)
            ax.axvline(0, color="0.86", lw=0.8, zorder=0)

            for term, row in wide.iterrows():
                px = float(row["Baseline"])
                py = float(row["Region fixed effects"])
                color = term_colors.get(term, "0.35")
                ax.scatter(px, py, s=42, color=color, edgecolor="white", linewidth=0.6, zorder=3)
                dx = 5 if py >= px else -5
                ha = "left" if dx > 0 else "right"
                ax.annotate(
                    short_labels.get(term, term),
                    xy=(px, py),
                    xytext=(dx, 4),
                    textcoords="offset points",
                    ha=ha,
                    va="bottom",
                    fontsize=6.8,
                    color=color,
                    clip_on=False,
                )

            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.set_aspect("equal", adjustable="box")
            if row_idx == 0:
                ax.set_title(conversion.replace("_", " "))
            if col_idx == 0:
                ax.set_ylabel(f"{spec['analysis'].replace(' model', '')}\nRegion-FE model estimate (%)")
            else:
                ax.set_ylabel("")
            ax.set_xlabel("Baseline model estimate (%)" if row_idx == len(model_specs) - 1 else "")
            ax.grid(color="0.92", lw=0.6)

    legend_handles = [
        mpl.lines.Line2D([0], [0], marker="o", color="none", markerfacecolor="#C44E52", markeredgecolor="white", markersize=5, label="Heat / high heat"),
        mpl.lines.Line2D([0], [0], marker="o", color="none", markerfacecolor="#4C72B0", markeredgecolor="white", markersize=5, label="Flood / high flood"),
        mpl.lines.Line2D([0], [0], marker="o", color="none", markerfacecolor="#55A868", markeredgecolor="white", markersize=5, label="Interaction / joint high"),
        mpl.lines.Line2D([0], [0], marker="o", color="none", markerfacecolor="#8172B2", markeredgecolor="white", markersize=5, label="EDI"),
        mpl.lines.Line2D([0], [0], color="0.45", lw=0.9, ls="--", label="No change"),
    ]
    fig.legend(
        handles=legend_handles,
        frameon=False,
        ncol=5,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.07),
        borderaxespad=0,
        handlelength=1.3,
        handletextpad=0.4,
        columnspacing=0.9,
        fontsize=8.0,
    )
    fig.suptitle(title, x=0.01, y=0.965, ha="left", fontweight="bold")
    # fig.text(0.01, 0.958, "Points compare baseline model estimates with estimates after adding African-region fixed effects", ha="left", va="top", fontsize=7.5)
    fig.tight_layout(rect=[0, 0.11, 1, 0.965], h_pad=0.85, w_pad=0.25)
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.show()


save_region_fe_scatter_plot(
    region_fe_plot,
    FIG_DIR / "sensitivity_region_fixed_effects_main_models",
    "Region fixed-effects sensitivity of Figure 2/3 main models",
    region_model_specs,
)


# Leave-one-country-out / influence robustness

In [ ]:
# Leave-one-country-out / influence robustness: refit Figure 2/3 main models after excluding one country at a time.
loo_model_specs = [
    {
        "analysis": "Continuous exposure model",
        "base_formula": continuous_formula,
        "terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"],
        "plot_terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z"],
    },
    {
        "analysis": "High exposure model",
        "base_formula": joint_formula,
        "terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"],
        "plot_terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z"],
    },
]

loo_rows = []
base_data = conversion_datasets()
for conversion, source_df in base_data.items():
    countries = sorted(source_df[COUNTRY_COL].dropna().astype(str).unique())
    for spec in loo_model_specs:
        model, model_df = fit_clustered(spec["base_formula"], source_df, "ratio_weight")
        before = len(loo_rows)
        collect_terms(
            model,
            loo_rows,
            spec["analysis"],
            conversion,
            "Full sample",
            spec["base_formula"],
            spec["terms"],
            len(model_df),
        )
        for row in loo_rows[before:]:
            row["dropped_country"] = "None"
            row["is_full_sample"] = True
            row["country_count"] = int(model_df[COUNTRY_COL].nunique())

        for country in countries:
            loo_df = source_df[source_df[COUNTRY_COL].astype(str) != country].copy()
            if loo_df[COUNTRY_COL].nunique() < 2:
                continue
            try:
                model, model_df = fit_clustered(spec["base_formula"], loo_df, "ratio_weight")
            except Exception as exc:
                loo_rows.append({
                    "analysis": spec["analysis"],
                    "conversion": conversion,
                    "variant": "Drop one country",
                    "formula": spec["base_formula"],
                    "term": "__fit_failed__",
                    "n": int(len(loo_df)),
                    "dropped_country": country,
                    "is_full_sample": False,
                    "country_count": int(loo_df[COUNTRY_COL].nunique()),
                    "error": str(exc),
                })
                continue
            before = len(loo_rows)
            collect_terms(
                model,
                loo_rows,
                spec["analysis"],
                conversion,
                "Drop one country",
                spec["base_formula"],
                spec["terms"],
                len(model_df),
            )
            for row in loo_rows[before:]:
                row["dropped_country"] = country
                row["is_full_sample"] = False
                row["country_count"] = int(model_df[COUNTRY_COL].nunique())

loo_sensitivity = pd.DataFrame(loo_rows)
loo_sensitivity.to_csv(TABLE_DIR / "sensitivity_leave_one_country_out_models.csv", index=False, encoding="utf-8-sig")

loo_valid = loo_sensitivity[loo_sensitivity["term"].ne("__fit_failed__")].copy()
loo_plot_terms = ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "high_heat", "high_flood", "joint_heat_flood"]
loo_valid = loo_valid[loo_valid["term"].isin(loo_plot_terms)].copy()

loo_full = loo_valid[loo_valid["is_full_sample"].eq(True)][
    ["analysis", "conversion", "term", "effect_pct", "ci_low_pct", "ci_high_pct"]
].rename(columns={
    "effect_pct": "full_effect_pct",
    "ci_low_pct": "full_ci_low_pct",
    "ci_high_pct": "full_ci_high_pct",
})
loo_drop = loo_valid[loo_valid["is_full_sample"].eq(False)].copy()

loo_summary = (
    loo_drop.groupby(["analysis", "conversion", "term"], as_index=False)
    .agg(
        n_countries=("dropped_country", "nunique"),
        min_effect_pct=("effect_pct", "min"),
        q25_effect_pct=("effect_pct", lambda x: x.quantile(0.25)),
        median_effect_pct=("effect_pct", "median"),
        q75_effect_pct=("effect_pct", lambda x: x.quantile(0.75)),
        max_effect_pct=("effect_pct", "max"),
    )
    .merge(loo_full, on=["analysis", "conversion", "term"], how="left")
)
loo_summary.to_csv(TABLE_DIR / "sensitivity_leave_one_country_out_summary.csv", index=False, encoding="utf-8-sig")

loo_influence = loo_drop.merge(
    loo_full[["analysis", "conversion", "term", "full_effect_pct"]],
    on=["analysis", "conversion", "term"],
    how="left",
)
loo_influence["abs_delta_effect_pct"] = (loo_influence["effect_pct"] - loo_influence["full_effect_pct"]).abs()
loo_top_influence = (
    loo_influence.sort_values("abs_delta_effect_pct", ascending=False)
    .groupby(["analysis", "conversion", "term"], as_index=False)
    .head(5)
)
loo_top_influence.to_csv(TABLE_DIR / "sensitivity_leave_one_country_out_top_influence.csv", index=False, encoding="utf-8-sig")


def save_leave_one_country_out_plot(summary_df, out_base, title, model_specs):
    conversion_order = ["infection_to_incidence_ratio", "incidence_to_mortality_ratio"]
    conversions = [c for c in conversion_order if c in set(summary_df["conversion"])]
    fig, axes = plt.subplots(len(model_specs), len(conversions), figsize=(10.2, 6.3), squeeze=False)
    range_color = "#8FBAD9"
    iqr_color = "#2C7FB8"
    median_color = "#1F78B4"

    for row_idx, spec in enumerate(model_specs):
        terms = spec["plot_terms"]
        y_lookup = {term: i for i, term in enumerate(terms)}
        for col_idx, conversion in enumerate(conversions):
            ax = axes[row_idx, col_idx]
            sub = summary_df[(summary_df["analysis"].eq(spec["analysis"])) & (summary_df["conversion"].eq(conversion)) & (summary_df["term"].isin(terms))].copy()
            sub["y"] = sub["term"].map(y_lookup)
            for _, row in sub.iterrows():
                y = row["y"]
                ax.hlines(y, row["min_effect_pct"], row["max_effect_pct"], color=range_color, lw=1.1, zorder=1)
                ax.hlines(y, row["q25_effect_pct"], row["q75_effect_pct"], color=iqr_color, lw=4.0, zorder=2)
                ax.scatter(row["median_effect_pct"], y, s=28, color=median_color, zorder=3)
                ax.scatter(row["full_effect_pct"], y, s=48, facecolors="none", edgecolors="black", linewidths=1.0, zorder=4)
            xmin = np.nanmin([sub["min_effect_pct"].min(), sub["full_ci_low_pct"].min(), 0])
            xmax = np.nanmax([sub["max_effect_pct"].max(), sub["full_ci_high_pct"].max(), 0])
            pad = max((xmax - xmin) * 0.12, 0.01)
            ax.set_xlim(xmin - pad, xmax + pad)
            ax.axvline(0, color="0.35", lw=0.8)
            ax.grid(axis="x", color="0.90", lw=0.6)
            ax.set_yticks(np.arange(len(terms)))
            ax.set_yticklabels([TERM_LABELS.get(term, term) for term in terms])
            if row_idx == 0:
                ax.set_title(conversion.replace("_", " "))
            if col_idx == 0:
                ax.set_ylabel(spec["analysis"].replace(" model", ""))
            ax.set_xlabel("Effect on log1p conversion ratio (%)")

    handles = [
        mpl.lines.Line2D([0], [0], color=range_color, lw=1.2, label="Min-max after dropping one country"),
        mpl.lines.Line2D([0], [0], color=iqr_color, lw=4.0, label="IQR after dropping one country"),
        mpl.lines.Line2D([0], [0], color=median_color, marker="o", lw=0, ms=4.5, label="Median leave-one-country-out"),
        mpl.lines.Line2D([0], [0], color="none", marker="o", markerfacecolor="none", markeredgecolor="black", ms=5.5, label="Full-sample main-text estimate"),
    ]
    fig.legend(
        handles=handles,
        frameon=False,
        ncol=2,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.07),
        borderaxespad=0,
        handlelength=1.8,
        columnspacing=1.2,
        handletextpad=0.45,
        fontsize=8.5
    )
    fig.suptitle(title, x=0.01, y=0.965, ha="left", fontweight="bold")
    # fig.text(0.01, 0.958, "Each interval summarizes refits after excluding one country at a time.", ha="left", va="top", fontsize=7.5)
    fig.tight_layout(rect=[0, 0.13, 1, 0.955], h_pad=1.0, w_pad=2.0)
    fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".png"), bbox_inches="tight", dpi=300)
    plt.show()


save_leave_one_country_out_plot(
    loo_summary,
    FIG_DIR / "sensitivity_leave_one_country_out_main_models",
    "Leave-one-country-out robustness of Figure 2/3 main models",
    loo_model_specs,
)


# Figure 4 contribution-classification threshold sensitivity


In [ ]:
# Figure 4 contribution-classification sensitivity: fixed contributions, no refitting.
from pathlib import Path

import numpy as np
import pandas as pd

def _figure4_source_path():
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        for root in [parent, parent / "analysis"]:
            candidate = root / "figure4" / "result4_dual_amplification_source_data.csv"
            if candidate.exists():
                return candidate
    raise FileNotFoundError("Export Figure 4 contribution source data before running this cell.")

f4_source_path = _figure4_source_path()
f4_table_dir = f4_source_path.parent.parent / "SI" / "tables"
f4_table_dir.mkdir(parents=True, exist_ok=True)
f4_data = pd.read_csv(f4_source_path, dtype={"adm2ID": str, "iso3": str})
f4_scores = ["i2i_mechanism_contribution_sd", "i2m_mechanism_contribution_sd"]
f4_required = ["orig_index", "adm2ID", "iso3", "region", "sl_pop", "dual_class", "clear_low_conversion", *f4_scores]
missing = set(f4_required) - set(f4_data.columns)
if missing:
    raise ValueError(f"Missing Figure 4 source columns: {sorted(missing)}")
if f4_data.empty or f4_data["orig_index"].isna().any() or f4_data["orig_index"].duplicated().any():
    raise ValueError("Figure 4 source must contain unique, nonmissing unit identifiers.")
for column in [*f4_scores, "sl_pop"]:
    f4_data[column] = pd.to_numeric(f4_data[column], errors="raise")
    if not np.isfinite(f4_data[column]).all():
        raise ValueError(f"Non-finite {column}; a fixed complete sample is required.")
if (f4_data["sl_pop"] < 0).any() or f4_data["sl_pop"].sum() <= 0:
    raise ValueError("Slum population must be nonnegative with a positive total.")
if f4_data[["iso3", "region"]].isna().any().any():
    raise ValueError("Missing country or region in Figure 4 source data.")

F4_CLASSES = ["Low conversion", "Infection-to-incidence only", "Incidence-to-mortality only", "Dual amplification", "Other subnational units"]

def classify_figure4_contributions(data, q):
    # Match Figure 4: unweighted quantiles on the joint analytic sample, inclusive ties.
    high = data[f4_scores].quantile(q)
    low = data[f4_scores].quantile(1 - q)
    i2i = data[f4_scores[0]] >= high.iloc[0]
    i2m = data[f4_scores[1]] >= high.iloc[1]
    low_both = (data[f4_scores] <= low).all(axis=1)
    labels = pd.Series(np.select(
        [i2i & i2m, i2i & ~i2m, ~i2i & i2m, low_both],
        ["Dual amplification", "Infection-to-incidence only", "Incidence-to-mortality only", "Low conversion"],
        default="Other subnational units",
    ), index=data.index)
    return labels, {
        "threshold_quantile": q, "low_threshold_quantile": 1 - q,
        "i2i_high_cut_sd": high.iloc[0], "i2m_high_cut_sd": high.iloc[1],
        "i2i_low_cut_sd": low.iloc[0], "i2m_low_cut_sd": low.iloc[1],
    }

f4_baseline, _ = classify_figure4_contributions(f4_data, 0.75)
saved_low = f4_data["clear_low_conversion"].astype(str).str.lower().map({"true": True, "false": False})
if saved_low.isna().any():
    raise ValueError("Unrecognized saved low-conversion flags.")
saved_classes = f4_data["dual_class"].replace({"Low/other conversion": "Other subnational units"}).copy()
saved_classes.loc[saved_low] = "Low conversion"
if not saved_classes.equals(f4_baseline):
    raise ValueError("Reclassified q75 does not reproduce the saved Figure 4 classes.")

def _pct(numerator, denominator):
    return 100 * numerator / denominator if denominator > 0 else np.nan

f4_summary_rows, f4_class_rows, f4_transition_rows, f4_unit_tables = [], [], [], []
f4_groups = [("Overall", "All", f4_data)]
for column, scope in [("region", "Region"), ("iso3", "Country")]:
    f4_groups.extend((scope, str(name), group) for name, group in f4_data.groupby(column, sort=True))

for q in [0.70, 0.75, 0.80]:
    labels, cuts = classify_figure4_contributions(f4_data, q)
    units = f4_data[["orig_index", "adm2ID", "iso3", "region", "sl_pop", *f4_scores]].copy()
    units["threshold_quantile"] = q
    units["baseline_class_q75"] = f4_baseline
    units["threshold_class"] = labels
    units["class_unchanged"] = labels.eq(f4_baseline)
    f4_unit_tables.append(units)
    for scope, name, group in f4_groups:
        current, base = labels.loc[group.index], f4_baseline.loc[group.index]
        pop = group["sl_pop"]
        total_pop = float(pop.sum())
        dual, base_dual = current.eq("Dual amplification"), base.eq("Dual amplification")
        overlap, union = dual & base_dual, dual | base_dual
        dual_pop, base_pop = float(pop[dual].sum()), float(pop[base_dual].sum())
        same = current.eq(base)
        f4_summary_rows.append({
            **cuts, "scope": scope, "group": name, "n_units": len(group), "slum_population": total_pop,
            "class_agreement_pct": _pct(same.sum(), len(group)),
            "population_weighted_agreement_pct": _pct(pop[same].sum(), total_pop),
            "dual_units": int(dual.sum()), "baseline_dual_units": int(base_dual.sum()),
            "dual_slum_population": dual_pop, "baseline_dual_slum_population": base_pop,
            "dual_population_share_pct": _pct(dual_pop, total_pop),
            "dual_population_change": dual_pop - base_pop,
            "dual_population_change_pct": _pct(dual_pop - base_pop, base_pop),
            "dual_population_share_change_pp": _pct(dual_pop - base_pop, total_pop),
            "dual_unit_jaccard_pct": _pct(overlap.sum(), union.sum()),
            "dual_population_jaccard_pct": _pct(pop[overlap].sum(), pop[union].sum()),
            "baseline_dual_units_retained_pct": _pct(overlap.sum(), base_dual.sum()),
            "baseline_dual_population_retained_pct": _pct(pop[overlap].sum(), base_pop),
            "dual_population_entered": float(pop[dual & ~base_dual].sum()),
            "dual_population_exited": float(pop[base_dual & ~dual].sum()),
        })
        for cls in F4_CLASSES:
            mask, base_mask = current.eq(cls), base.eq(cls)
            intersection, class_union = mask & base_mask, mask | base_mask
            f4_class_rows.append({
                "threshold_quantile": q, "scope": scope, "group": name, "class": cls,
                "n_units": int(mask.sum()), "unit_share_pct": _pct(mask.sum(), len(group)),
                "slum_population": float(pop[mask].sum()), "population_share_pct": _pct(pop[mask].sum(), total_pop),
                "baseline_units_retained_pct": _pct(intersection.sum(), base_mask.sum()),
                "unit_jaccard_pct": _pct(intersection.sum(), class_union.sum()),
            })
        if scope == "Overall":
            for from_class in F4_CLASSES:
                for to_class in F4_CLASSES:
                    mask = base.eq(from_class) & current.eq(to_class)
                    f4_transition_rows.append({
                        "threshold_quantile": q, "baseline_class_q75": from_class,
                        "threshold_class": to_class, "n_units": int(mask.sum()),
                        "slum_population": float(pop[mask].sum()),
                        "baseline_class_units_pct": _pct(mask.sum(), base.eq(from_class).sum()),
                    })

f4_threshold_summary = pd.DataFrame(f4_summary_rows)
f4_class_summary = pd.DataFrame(f4_class_rows)
f4_transitions = pd.DataFrame(f4_transition_rows)
f4_unit_classifications = pd.concat(f4_unit_tables, ignore_index=True)
f4_overall = f4_threshold_summary.query("scope == 'Overall'").copy()
# Check that relaxing the threshold can only expand the dual-amplification set.
assert f4_overall["dual_units"].is_monotonic_decreasing
assert f4_overall["dual_slum_population"].is_monotonic_decreasing
for _, sub in f4_class_summary.groupby(["threshold_quantile", "scope", "group"]):
    assert np.isclose(sub["unit_share_pct"].sum(), 100)
for _, sub in f4_transitions.groupby("threshold_quantile"):
    assert sub["n_units"].sum() == len(f4_data)
    assert np.isclose(sub["slum_population"].sum(), f4_data["sl_pop"].sum())

# Geographic stability: rank by absolute dual-class population, not within-country prevalence.
f4_quantiles = [0.70, 0.75, 0.80]
f4_country_pop = f4_threshold_summary.query("scope == 'Country'").pivot(
    index="group", columns="threshold_quantile", values="dual_slum_population"
).reindex(columns=f4_quantiles)
f4_region_pop = f4_threshold_summary.query("scope == 'Region'").pivot(
    index="group", columns="threshold_quantile", values="dual_slum_population"
).reindex(columns=f4_quantiles)
f4_totals = f4_overall.set_index("threshold_quantile")["dual_slum_population"].reindex(f4_quantiles)
assert np.allclose(f4_country_pop.sum(), f4_totals)
assert np.allclose(f4_region_pop.sum(), f4_totals)

# Tied populations receive the same minimum rank; include all ties at rank 10.
# Countries with zero dual-class population are not ranked.
f4_country_ranks = f4_country_pop.where(f4_country_pop > 0).rank(ascending=False, method="min")
f4_top_sets = {q: set(f4_country_ranks.index[f4_country_ranks[q] <= 10]) for q in f4_quantiles}
f4_selected = set().union(*f4_top_sets.values())
f4_country_order = f4_country_pop.loc[sorted(f4_selected)].sort_values(
    0.75, ascending=False, kind="stable"
).index
f4_names = f4_data[["iso3", "country_name"]].drop_duplicates()
if f4_names["iso3"].duplicated().any():
    raise ValueError("Country names are inconsistent within an ISO3 code.")
f4_name_map = f4_names.set_index("iso3")["country_name"]
f4_country_table = pd.DataFrame({
    "Country": f4_country_order.map(f4_name_map),
})
for q in f4_quantiles:
    label = f"{q:.0%}"
    f4_country_table[f"Rank at {label}"] = pd.array(
        f4_country_ranks.loc[f4_country_order, q].to_numpy(), dtype="Int64"
    )
f4_region_order = ["Western", "Central", "Eastern", "Northern", "Southern"]
if set(f4_region_pop.index) != set(f4_region_order):
    raise ValueError("Expected the five Figure 4 regions.")
f4_region_pop = f4_region_pop.reindex(f4_region_order)
# Shares use the total dual-class population across all regions at each threshold.
f4_region_shares = f4_region_pop.div(f4_totals, axis="columns") * 100
assert np.allclose(f4_region_shares.sum(), 100)
f4_region_table = pd.DataFrame({"Region": [r + " Africa" for r in f4_region_order]})
for q in f4_quantiles:
    f4_region_table[f"Share of all dual population at {q:.0%} (%)"] = f4_region_shares[q].to_numpy().round(2)
f4_geographic_tables = {
    "country_priority": f4_country_table,
    "regional_composition": f4_region_table,
}
for suffix, table in f4_geographic_tables.items():
    path = f4_table_dir / f"sensitivity_figure4_contribution_threshold_{suffix}.csv"
    table.to_csv(path, index=False, encoding="utf-8-sig", na_rep="NA")
    try:
        from IPython.display import display
    except ImportError:
        print(table.to_string(index=False))
    else:
        display(table)

print("Countries shown: union of top-10 population ranks across thresholds; ties included.")
print("Ranks use unrounded dual-class population; zero-population countries are unranked (NA).")
print("Regional shares use all dual-class population at each threshold, not regional slum population.")
for q in [0.70, 0.80]:
    shared = f4_top_sets[q] & f4_top_sets[0.75]
    print(f"{q:.0%}: {len(shared)}/{len(f4_top_sets[0.75])} baseline top countries retained; "
          f"alternative top set has {len(f4_top_sets[q])} countries.")
    max_region = (f4_region_shares[q] - f4_region_shares[0.75]).abs().idxmax()
    delta = f4_region_shares.loc[max_region, q] - f4_region_shares.loc[max_region, 0.75]
    print(f"Largest regional composition change: {max_region} Africa, {delta:+.2f} pp.")


# Additional tabular sensitivity analyses


In [ ]:
# Additional tabular sensitivity analyses.
# These are new checks, not table reformatting of the five sensitivity figures above.
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)

# Self-contained setup for this table-analysis block.  These definitions mirror the
# shared setup used by the figure-based sensitivity cells, so this cell still works
# if earlier exploratory/table cells were deleted or not run in the current kernel.
if "PROJECT_ROOT" not in globals():
    def find_project_root():
        cwd = Path.cwd().resolve()
        for candidate in [cwd, *cwd.parents]:
            shp = candidate / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
            if shp.exists():
                return candidate
        for candidate in [cwd, *cwd.parents]:
            nested = candidate / "analysis"
            shp = nested / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
            if shp.exists():
                return nested
        raise FileNotFoundError("Cannot locate analysis project root.")

    PROJECT_ROOT = find_project_root()

if "TABLE_DIR" not in globals():
    TABLE_DIR = PROJECT_ROOT / "supplementary" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if "DATA_PATH" not in globals():
    DATA_PATH = PROJECT_ROOT / "data" / "overall" / "admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"

BASE_HEAT_COL = globals().get("BASE_HEAT_COL", "HE_perpop")
BASE_FLOOD_COL = globals().get("BASE_FLOOD_COL", "sl_sevexp")
EDI_COL = globals().get("EDI_COL", "EDI_qmean")
COUNTRY_COL = globals().get("COUNTRY_COL", "iso3")
POP_COL = globals().get("POP_COL", "sl_pop")

if "raw_gdf" not in globals():
    raw_gdf = gpd.read_file(DATA_PATH)
    raw_gdf[COUNTRY_COL] = raw_gdf[COUNTRY_COL].astype(str)

if "zscore" not in globals():
    def zscore(x):
        x = pd.to_numeric(x, errors="coerce")
        std = x.std(ddof=0)
        if not np.isfinite(std) or std == 0:
            return x * np.nan
        return (x - x.mean()) / std

if "prepare_ratio" not in globals():
    def prepare_ratio(numerator_col, denominator_col, conversion, heat_col=BASE_HEAT_COL, flood_col=BASE_FLOOD_COL, threshold_q=0.75):
        d = raw_gdf.copy()
        d["ratio_numer"] = pd.to_numeric(d[numerator_col], errors="coerce")
        d["ratio_denom"] = pd.to_numeric(d[denominator_col], errors="coerce")
        d["ratio_weight"] = d["ratio_denom"]
        d["conversion_ratio"] = d["ratio_numer"] / d["ratio_denom"]
        invalid = (d["ratio_denom"] <= 0) | (d["ratio_numer"] < 0) | (d["conversion_ratio"] < 0)
        d.loc[invalid, ["conversion_ratio", "ratio_weight"]] = np.nan
        d["log_ratio"] = np.log1p(d["conversion_ratio"])
        d["heat_z"] = zscore(d[heat_col])
        d["flood_z"] = zscore(d[flood_col])
        d["edi_z"] = zscore(d[EDI_COL])
        d["logpop_z"] = zscore(np.log1p(d["ratio_weight"]))
        d["slpop_weight"] = pd.to_numeric(d[POP_COL], errors="coerce")
        heat_values = pd.to_numeric(d[heat_col], errors="coerce")
        flood_values = pd.to_numeric(d[flood_col], errors="coerce")
        d["high_heat"] = (heat_values >= heat_values.quantile(threshold_q)).astype(int)
        d["high_flood"] = (flood_values >= flood_values.quantile(threshold_q)).astype(int)
        d["joint_heat_flood"] = ((d["high_heat"] == 1) & (d["high_flood"] == 1)).astype(int)
        d["conversion"] = conversion
        needed = ["log_ratio", "conversion_ratio", "ratio_weight", "heat_z", "flood_z", "edi_z", "logpop_z", COUNTRY_COL]
        return d.dropna(subset=needed).copy()

if "conversion_datasets" not in globals():
    def conversion_datasets(heat_col=BASE_HEAT_COL, flood_col=BASE_FLOOD_COL, threshold_q=0.75):
        return {
            "infection_to_incidence_ratio": prepare_ratio("sl_pfinc", "sl_pfinf", "infection_to_incidence_ratio", heat_col, flood_col, threshold_q),
            "incidence_to_mortality_ratio": prepare_ratio("sl_pfmort", "sl_pfinc", "incidence_to_mortality_ratio", heat_col, flood_col, threshold_q),
        }


ADDITIONAL_TERM_LABELS = {
    "heat_z": "Heat exposure",
    "flood_z": "Flood exposure",
    "heat_z:flood_z": "Heat x flood",
    "I(heat_z ** 2)": "Heat exposure squared",
    "I(flood_z ** 2)": "Flood exposure squared",
    "edi_z": "EDI",
    "high_heat": "High heat",
    "high_flood": "High flood",
    "joint_heat_flood": "High heat and high flood",
    "logpop_z": "Log denominator population",
}
ADDITIONAL_CONVERSION_LABELS = {
    "infection_to_incidence_ratio": "Infection-to-incidence",
    "incidence_to_mortality_ratio": "Incidence-to-mortality",
}


def _label_additional_rows(df):
    out = df.copy()
    out["conversion_label"] = out["conversion"].map(ADDITIONAL_CONVERSION_LABELS).fillna(out["conversion"])
    out["term_label"] = out["term"].map(ADDITIONAL_TERM_LABELS).fillna(out["term"])
    return out


def _effect_ci(row):
    if pd.isna(row.get("effect_pct")) or pd.isna(row.get("ci_low_pct")) or pd.isna(row.get("ci_high_pct")):
        return np.nan
    return f"{row['effect_pct']:.2f} ({row['ci_low_pct']:.2f}, {row['ci_high_pct']:.2f})"


def _fit_wls_country_cluster(formula, data, weight_col="ratio_weight"):
    model_df = data.dropna(subset=[weight_col, COUNTRY_COL]).copy()
    model_df = model_df[pd.to_numeric(model_df[weight_col], errors="coerce") > 0].copy()
    fit = smf.wls(formula, data=model_df, weights=model_df[weight_col]).fit(
        cov_type="cluster", cov_kwds={"groups": model_df[COUNTRY_COL]}
    )
    return fit, model_df


def _collect_extra_terms(model, rows, analysis, conversion, variant, formula, terms, model_df, extra=None):
    extra = extra or {}
    for term in terms:
        if term not in model.params.index:
            continue
        coef = float(model.params[term])
        se = float(model.bse[term])
        row = {
            "analysis": analysis,
            "conversion": conversion,
            "variant": variant,
            "formula": formula,
            "term": term,
            "n": int(len(model_df)),
            "country_count": int(model_df[COUNTRY_COL].nunique()),
            "coef": coef,
            "se": se,
            "ci_low": coef - 1.96 * se,
            "ci_high": coef + 1.96 * se,
            "pval": float(model.pvalues[term]),
            "effect_pct": 100 * (np.exp(coef) - 1),
            "ci_low_pct": 100 * (np.exp(coef - 1.96 * se) - 1),
            "ci_high_pct": 100 * (np.exp(coef + 1.96 * se) - 1),
        }
        row.update(extra)
        rows.append(row)


base_data = conversion_datasets()

# Write these outputs to the directory that already contains the five sensitivity-analysis CSVs.
_candidate_table_dirs = [
    PROJECT_ROOT / "supplementary" / "tables",
    Path.cwd() / "tables",
    Path.cwd() / "SI" / "tables",
    PROJECT_ROOT / "supplementary" / "tables",
    PROJECT_ROOT / "supplementary" / "tables" if "PROJECT_ROOT" in globals() else PROJECT_ROOT / "supplementary" / "tables",
    TABLE_DIR,
]
ADDITIONAL_TABLE_DIR = None
for _candidate in _candidate_table_dirs:
    _candidate = Path(_candidate)
    if (_candidate / "sensitivity_weight_models.csv").exists():
        ADDITIONAL_TABLE_DIR = _candidate
        break
if ADDITIONAL_TABLE_DIR is None:
    ADDITIONAL_TABLE_DIR = Path(TABLE_DIR)
ADDITIONAL_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# New analysis 1: adjustment-set / functional-form sensitivity.
adjustment_specs = [
    {
        "analysis": "Continuous exposure model",
        "variant": "Baseline denominator WLS",
        "formula": "log_ratio ~ heat_z + flood_z + heat_z:flood_z + edi_z + logpop_z",
        "terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"],
    },
    {
        "analysis": "Continuous exposure model",
        "variant": "No log-population control",
        "formula": "log_ratio ~ heat_z + flood_z + heat_z:flood_z + edi_z",
        "terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z"],
    },
    {
        "analysis": "Continuous exposure model",
        "variant": "Quadratic heat/flood terms",
        "formula": "log_ratio ~ heat_z + flood_z + heat_z:flood_z + I(heat_z ** 2) + I(flood_z ** 2) + edi_z + logpop_z",
        "terms": ["heat_z", "flood_z", "heat_z:flood_z", "I(heat_z ** 2)", "I(flood_z ** 2)", "edi_z", "logpop_z"],
    },
    {
        "analysis": "Continuous exposure model",
        "variant": "Country fixed effects",
        "formula": "log_ratio ~ heat_z + flood_z + heat_z:flood_z + edi_z + logpop_z + C(iso3)",
        "terms": ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"],
    },
    {
        "analysis": "High exposure model",
        "variant": "Baseline denominator WLS",
        "formula": "log_ratio ~ high_heat + high_flood + joint_heat_flood + edi_z + logpop_z",
        "terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"],
    },
    {
        "analysis": "High exposure model",
        "variant": "No log-population control",
        "formula": "log_ratio ~ high_heat + high_flood + joint_heat_flood + edi_z",
        "terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z"],
    },
    {
        "analysis": "High exposure model",
        "variant": "Country fixed effects",
        "formula": "log_ratio ~ high_heat + high_flood + joint_heat_flood + edi_z + logpop_z + C(iso3)",
        "terms": ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"],
    },
]

adjustment_rows = []
for conversion, source_df in base_data.items():
    for spec in adjustment_specs:
        try:
            model, model_df = _fit_wls_country_cluster(spec["formula"], source_df)
            _collect_extra_terms(
                model, adjustment_rows, spec["analysis"], conversion, spec["variant"],
                spec["formula"], spec["terms"], model_df,
                {"sensitivity_question": "Adjustment set and functional form"},
            )
        except Exception as exc:
            adjustment_rows.append({
                "analysis": spec["analysis"],
                "conversion": conversion,
                "variant": spec["variant"],
                "formula": spec["formula"],
                "term": "__fit_failed__",
                "error": str(exc),
                "sensitivity_question": "Adjustment set and functional form",
            })

adjustment_sensitivity = _label_additional_rows(pd.DataFrame(adjustment_rows))
adjustment_sensitivity["effect_pct_95ci"] = adjustment_sensitivity.apply(_effect_ci, axis=1)
adjustment_sensitivity.to_csv(ADDITIONAL_TABLE_DIR / "sensitivity_new_adjustment_set_models.csv", index=False, encoding="utf-8-sig")

# New analysis 2: outcome-tail and denominator sample-restriction sensitivity.
def _restriction_variants(df):
    ratio_q95 = float(df["conversion_ratio"].quantile(0.95))
    ratio_q99 = float(df["conversion_ratio"].quantile(0.99))
    denom_q05 = float(df["ratio_weight"].quantile(0.05))
    return [
        ("Full analytic sample", df.copy(), "No additional restriction"),
        ("Exclude top 1% conversion ratios", df[df["conversion_ratio"] <= ratio_q99].copy(), f"conversion_ratio <= p99 ({ratio_q99:.6g})"),
        ("Exclude top 5% conversion ratios", df[df["conversion_ratio"] <= ratio_q95].copy(), f"conversion_ratio <= p95 ({ratio_q95:.6g})"),
        ("Exclude bottom 5% denominators", df[df["ratio_weight"] >= denom_q05].copy(), f"ratio_weight >= p05 ({denom_q05:.6g})"),
        (
            "Exclude top 1% ratios and bottom 5% denominators",
            df[(df["conversion_ratio"] <= ratio_q99) & (df["ratio_weight"] >= denom_q05)].copy(),
            f"conversion_ratio <= p99 and ratio_weight >= p05",
        ),
    ]

restriction_model_specs = [
    ("Continuous exposure model", "log_ratio ~ heat_z + flood_z + heat_z:flood_z + edi_z + logpop_z", ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"]),
    ("High exposure model", "log_ratio ~ high_heat + high_flood + joint_heat_flood + edi_z + logpop_z", ["high_heat", "high_flood", "joint_heat_flood", "edi_z", "logpop_z"]),
]

restriction_rows = []
for conversion, source_df in base_data.items():
    n_full = len(source_df)
    for variant, restricted_df, rule in _restriction_variants(source_df):
        for analysis, formula, terms in restriction_model_specs:
            try:
                model, model_df = _fit_wls_country_cluster(formula, restricted_df)
                _collect_extra_terms(
                    model, restriction_rows, analysis, conversion, variant, formula, terms, model_df,
                    {
                        "sensitivity_question": "Outcome tail and denominator restriction",
                        "restriction_rule": rule,
                        "n_full": int(n_full),
                        "retained_pct": 100 * len(model_df) / n_full if n_full else np.nan,
                    },
                )
            except Exception as exc:
                restriction_rows.append({
                    "analysis": analysis,
                    "conversion": conversion,
                    "variant": variant,
                    "formula": formula,
                    "term": "__fit_failed__",
                    "error": str(exc),
                    "sensitivity_question": "Outcome tail and denominator restriction",
                    "restriction_rule": rule,
                    "n_full": int(n_full),
                })

restriction_sensitivity = _label_additional_rows(pd.DataFrame(restriction_rows))
restriction_sensitivity["effect_pct_95ci"] = restriction_sensitivity.apply(_effect_ci, axis=1)
restriction_sensitivity.to_csv(ADDITIONAL_TABLE_DIR / "sensitivity_new_outcome_tail_restriction_models.csv", index=False, encoding="utf-8-sig")

# New analysis 3: inference sensitivity to covariance estimator.
def _fit_wls_covariance(formula, data, covariance_variant):
    model_df = data.dropna(subset=["ratio_weight", COUNTRY_COL]).copy()
    model_df = model_df[pd.to_numeric(model_df["ratio_weight"], errors="coerce") > 0].copy()
    base = smf.wls(formula, data=model_df, weights=model_df["ratio_weight"])
    if covariance_variant == "Country-clustered SE":
        fit = base.fit(cov_type="cluster", cov_kwds={"groups": model_df[COUNTRY_COL]})
    elif covariance_variant == "HC3 robust SE":
        fit = base.fit(cov_type="HC3")
    elif covariance_variant == "HC1 robust SE":
        fit = base.fit(cov_type="HC1")
    elif covariance_variant == "Classical WLS SE":
        fit = base.fit()
    else:
        raise ValueError(covariance_variant)
    return fit, model_df

covariance_variants = ["Country-clustered SE", "HC3 robust SE", "HC1 robust SE", "Classical WLS SE"]
inference_rows = []
for conversion, source_df in base_data.items():
    for analysis, formula, terms in restriction_model_specs:
        for covariance_variant in covariance_variants:
            try:
                model, model_df = _fit_wls_covariance(formula, source_df, covariance_variant)
                _collect_extra_terms(
                    model, inference_rows, analysis, conversion, covariance_variant, formula, terms, model_df,
                    {"sensitivity_question": "Inference covariance estimator", "covariance_estimator": covariance_variant},
                )
            except Exception as exc:
                inference_rows.append({
                    "analysis": analysis,
                    "conversion": conversion,
                    "variant": covariance_variant,
                    "formula": formula,
                    "term": "__fit_failed__",
                    "error": str(exc),
                    "sensitivity_question": "Inference covariance estimator",
                    "covariance_estimator": covariance_variant,
                })

inference_sensitivity = _label_additional_rows(pd.DataFrame(inference_rows))
inference_sensitivity["effect_pct_95ci"] = inference_sensitivity.apply(_effect_ci, axis=1)
inference_sensitivity.to_csv(ADDITIONAL_TABLE_DIR / "sensitivity_new_inference_covariance_models.csv", index=False, encoding="utf-8-sig")

# Compact workbook for the three new analyses.
xlsx_path = ADDITIONAL_TABLE_DIR / "sensitivity_new_additional_analyses.xlsx"
try:
    with pd.ExcelWriter(xlsx_path) as writer:
        adjustment_sensitivity.to_excel(writer, sheet_name="adjustment_set", index=False)
        restriction_sensitivity.to_excel(writer, sheet_name="outcome_tail_restriction", index=False)
        inference_sensitivity.to_excel(writer, sheet_name="inference_covariance", index=False)
except Exception as exc:
    print(f"Skipped XLSX export: {exc}")

print("Saved new sensitivity analyses:")
print(f"- sensitivity_new_adjustment_set_models.csv: {adjustment_sensitivity.shape[0]} rows")
print(f"- sensitivity_new_outcome_tail_restriction_models.csv: {restriction_sensitivity.shape[0]} rows")
print(f"- sensitivity_new_inference_covariance_models.csv: {inference_sensitivity.shape[0]} rows")
print(f"- sensitivity_new_additional_analyses.xlsx")
print(f"Output directory: {ADDITIONAL_TABLE_DIR}")


In [ ]:
# SI-ready manuscript-style tables for the additional sensitivity analyses.
# The three tables have distinct structures, following a main-table + compact robustness-table style.
from pathlib import Path
import numpy as np
import pandas as pd

try:
    ADDITIONAL_TABLE_DIR
except NameError:
    candidates = [
        PROJECT_ROOT / "supplementary" / "tables",
        Path.cwd() / "tables",
        Path.cwd() / "SI" / "tables",
        PROJECT_ROOT / "supplementary" / "tables",
    ]
    ADDITIONAL_TABLE_DIR = next((p for p in candidates if (p / "sensitivity_new_adjustment_set_models.csv").exists()), candidates[0])

CONVERSION_LABEL = {
    "infection_to_incidence_ratio": "Infection-to-incidence",
    "incidence_to_mortality_ratio": "Incidence-to-mortality",
}
MODEL_LABEL = {
    "Continuous exposure model": "Continuous exposure",
    "High exposure model": "High exposure",
}
TERM_LABEL = {
    "heat_z": "Heat",
    "flood_z": "Flood",
    "heat_z:flood_z": "Heat x flood",
    "edi_z": "EDI",
    "high_heat": "High heat",
    "high_flood": "High flood",
    "joint_heat_flood": "High heat and high flood",
}


def read_new_table(name):
    return pd.read_csv(ADDITIONAL_TABLE_DIR / name)


def fmt_effect_ci(row):
    if pd.isna(row.get("effect_pct")) or pd.isna(row.get("ci_low_pct")) or pd.isna(row.get("ci_high_pct")):
        return "NA"
    return f"{row['effect_pct']:.2f} ({row['ci_low_pct']:.2f}, {row['ci_high_pct']:.2f})"


def fmt_range(vals):
    vals = pd.to_numeric(vals, errors="coerce").dropna()
    if vals.empty:
        return "NA"
    return f"{vals.min():.2f} to {vals.max():.2f}"


def ci_excludes_zero(row):
    if pd.isna(row.get("ci_low_pct")) or pd.isna(row.get("ci_high_pct")):
        return False
    return (row["ci_low_pct"] > 0) or (row["ci_high_pct"] < 0)


def sign_symbol(x):
    if pd.isna(x):
        return "NA"
    if x > 0:
        return "+"
    if x < 0:
        return "-"
    return "0"


def write_table_bundle(df, stem, caption, note=None):
    csv_path = ADDITIONAL_TABLE_DIR / f"{stem}.csv"
    xlsx_path = ADDITIONAL_TABLE_DIR / f"{stem}.xlsx"
    html_path = ADDITIONAL_TABLE_DIR / f"{stem}.html"
    md_path = ADDITIONAL_TABLE_DIR / f"{stem}.md"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    try:
        df.to_excel(xlsx_path, index=False)
    except Exception as exc:
        print(f"Skipped XLSX export for {stem}: {exc}")
    html = "<html><head><meta charset='utf-8'><style>body{font-family:Arial,sans-serif;font-size:10pt;} table{border-collapse:collapse;width:100%;} th,td{border:1px solid #999;padding:4px;vertical-align:top;} th{background:#f2f2f2;}</style></head><body>"
    html += f"<p><b>{caption}</b></p>"
    if note:
        html += f"<p><i>Note.</i> {note}</p>"
    html += df.to_html(index=False, escape=False)
    html += "</body></html>"
    html_path.write_text(html, encoding="utf-8")
    cols = list(df.columns)
    lines = [caption, ""]
    if note:
        lines.extend([f"Note. {note}", ""])
    lines.append("| " + " | ".join(cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in df.iterrows():
        vals = [str(row[c]).replace("|", "/") for c in cols]
        lines.append("| " + " | ".join(vals) + " |")
    md_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return csv_path


adjustment = read_new_table("sensitivity_new_adjustment_set_models.csv")
restriction = read_new_table("sensitivity_new_outcome_tail_restriction_models.csv")
inference = read_new_table("sensitivity_new_inference_covariance_models.csv")

# Table 1: analytical scope and interpretation for all seven sensitivity analyses.
# This table is a manuscript-facing inventory: five existing figure-based analyses plus
# two new table-based analyses. It is not a coefficient matrix.
table1 = pd.DataFrame([
    {
        "Sensitivity analysis": "Weighting specification",
        "Status": "Completed; figure-based",
        "Display item": "Sensitivity figure: weighting specification",
        "Question addressed": "Whether coefficient estimates depend on using unweighted OLS, denominator-weighted WLS, square-root-denominator WLS or capped-denominator WLS.",
        "Primary readout": "Effect estimates and 95% CIs across weighting specifications for the continuous and high-exposure models.",
        "Source-data table": "sensitivity_weight_models.csv",
    },
    {
        "Sensitivity analysis": "Exposure-definition sensitivity",
        "Status": "Completed; figure-based",
        "Display item": "Sensitivity figures: exposure-definition heatmaps and delta forest plot",
        "Question addressed": "Whether results change when alternative heat metrics and flood metrics replace the baseline HE_perpop and sl_sevexp definitions.",
        "Primary readout": "Coefficient surfaces and changes from the baseline exposure-definition model.",
        "Source-data table": "sensitivity_exposure_definition_models.csv; sensitivity_exposure_definition_delta_summary.csv",
    },
    {
        "Sensitivity analysis": "High-exposure threshold sensitivity",
        "Status": "Completed; figure-based",
        "Display item": "Sensitivity figure: high-exposure threshold model",
        "Question addressed": "Whether high-exposure model conclusions depend on the quantile threshold used to define high heat, high flood and joint high exposure.",
        "Primary readout": "Direction, magnitude and interval support across alternative high-exposure thresholds.",
        "Source-data table": "sensitivity_high_exposure_threshold_models.csv; sensitivity_high_exposure_threshold_direction_summary.csv",
    },
    {
        "Sensitivity analysis": "Region fixed-effects sensitivity",
        "Status": "Completed; figure-based",
        "Display item": "Sensitivity figure: region fixed-effects models",
        "Question addressed": "Whether main model terms are retained after adding broad African-region fixed effects.",
        "Primary readout": "Baseline versus region-fixed-effects coefficient estimates and 95% CIs.",
        "Source-data table": "sensitivity_region_fixed_effects_models.csv",
    },
    {
        "Sensitivity analysis": "Leave-one-country-out influence",
        "Status": "Completed; figure-based",
        "Display item": "Sensitivity figure: leave-one-country-out models",
        "Question addressed": "Whether model conclusions are driven by excluding any single country from the analytic sample.",
        "Primary readout": "Full-sample estimate compared with leave-one-country-out median, interquartile range and min-max range.",
        "Source-data table": "sensitivity_leave_one_country_out_models.csv; sensitivity_leave_one_country_out_summary.csv; sensitivity_leave_one_country_out_top_influence.csv",
    },
    {
        "Sensitivity analysis": "Outcome-tail and denominator restriction",
        "Status": "Completed; table-based",
        "Display item": "Supplementary table: selected sample-restriction estimates",
        "Question addressed": "Whether selected estimates are driven by extreme conversion ratios or small denominator populations.",
        "Primary readout": "Full-sample estimates compared with exclusions of top 1% ratios, top 5% ratios, bottom 5% denominators and the combined restriction.",
        "Source-data table": "sensitivity_new_outcome_tail_restriction_models.csv",
    },
    {
        "Sensitivity analysis": "Covariance-estimator sensitivity",
        "Status": "Completed; table-based",
        "Display item": "Supplementary table: covariance-estimator interval robustness",
        "Question addressed": "Whether interval-based conclusions for baseline-supported terms depend on the standard-error estimator.",
        "Primary readout": "Country-clustered 95% CIs compared with HC3, HC1 and classical WLS 95% CIs.",
        "Source-data table": "sensitivity_new_inference_covariance_models.csv",
    },
])

# Table 2: outcome-tail and denominator-restriction sensitivity only.
# This table avoids repeating adjustment-set or covariance information. Each column is a different
# sample restriction, so the table directly reports whether extreme ratios or small denominators drive estimates.
selected_terms = [
    ("Continuous exposure model", "infection_to_incidence_ratio", "edi_z"),
    ("Continuous exposure model", "infection_to_incidence_ratio", "heat_z:flood_z"),
    ("High exposure model", "infection_to_incidence_ratio", "high_heat"),
    ("High exposure model", "infection_to_incidence_ratio", "joint_heat_flood"),
    ("High exposure model", "infection_to_incidence_ratio", "edi_z"),
    ("Continuous exposure model", "incidence_to_mortality_ratio", "flood_z"),
    ("High exposure model", "incidence_to_mortality_ratio", "high_flood"),
    ("High exposure model", "incidence_to_mortality_ratio", "joint_heat_flood"),
]
restriction_variants = [
    ("Full analytic sample", "Full sample, % (95% CI)"),
    ("Exclude top 1% conversion ratios", "Exclude top 1% ratios, % (95% CI)"),
    ("Exclude top 5% conversion ratios", "Exclude top 5% ratios, % (95% CI)"),
    ("Exclude bottom 5% denominators", "Exclude bottom 5% denominators, % (95% CI)"),
    ("Exclude top 1% ratios and bottom 5% denominators", "Combined restriction, % (95% CI)"),
]

restriction_rows = []
for analysis, conversion, term in selected_terms:
    sub = restriction[
        restriction["analysis"].eq(analysis)
        & restriction["conversion"].eq(conversion)
        & restriction["term"].eq(term)
    ].copy()
    if sub.empty:
        continue
    base = sub[sub["variant"].eq("Full analytic sample")]
    if base.empty:
        continue
    base_row = base.iloc[0]
    base_sign = sign_symbol(base_row["effect_pct"])
    row = {
        "Model": MODEL_LABEL.get(analysis, analysis),
        "Conversion": CONVERSION_LABEL.get(conversion, conversion),
        "Term": TERM_LABEL.get(term, term),
    }
    n_values = []
    retained_values = []
    for variant, col in restriction_variants:
        v = sub[sub["variant"].eq(variant)]
        if v.empty:
            row[col] = "NA"
            continue
        rr = v.iloc[0]
        row[col] = fmt_effect_ci(rr)
        n_values.append(int(rr["n"]))
        if "retained_pct" in rr.index and pd.notna(rr["retained_pct"]):
            retained_values.append(float(rr["retained_pct"]))
    same_sign = int((sub["effect_pct"].map(sign_symbol) == base_sign).sum())
    ci_nonzero = int(sub.apply(ci_excludes_zero, axis=1).sum())
    row["N range"] = f"{min(n_values):,} to {max(n_values):,}" if n_values else "NA"
    row["Retained sample range"] = f"{min(retained_values):.1f}% to {max(retained_values):.1f}%" if retained_values else "NA"
    row["Same direction as full sample"] = f"{same_sign}/{len(sub)}"
    row["CI excludes zero"] = f"{ci_nonzero}/{len(sub)}"
    restriction_rows.append(row)

table2 = pd.DataFrame(restriction_rows)

# Table 3: covariance-estimator interval robustness for the same selected baseline-significant terms.
inf_rows = []
for analysis, conversion, term in selected_terms:
    sub = inference[
        inference["analysis"].eq(analysis)
        & inference["conversion"].eq(conversion)
        & inference["term"].eq(term)
    ].copy()
    if sub.empty:
        continue
    base = sub[sub["variant"].eq("Country-clustered SE")]
    if base.empty or not ci_excludes_zero(base.iloc[0]):
        continue
    row = {
        "Model": MODEL_LABEL.get(analysis, analysis),
        "Conversion": CONVERSION_LABEL.get(conversion, conversion),
        "Term": TERM_LABEL.get(term, term),
        "Point estimate, %": f"{base.iloc[0]['effect_pct']:.2f}",
    }
    for variant, col in [
        ("Country-clustered SE", "Country-clustered 95% CI"),
        ("HC3 robust SE", "HC3 95% CI"),
        ("HC1 robust SE", "HC1 95% CI"),
        ("Classical WLS SE", "Classical WLS 95% CI"),
    ]:
        v = sub[sub["variant"].eq(variant)]
        if v.empty:
            row[col] = "NA"
        else:
            rr = v.iloc[0]
            row[col] = f"{rr['ci_low_pct']:.2f} to {rr['ci_high_pct']:.2f}"
    row["All intervals exclude zero"] = "Yes" if sub.apply(ci_excludes_zero, axis=1).all() else "No"
    inf_rows.append(row)

table3 = pd.DataFrame(inf_rows)

captions = {
    "sensitivity_new_si_table1_analysis_scope": "Supplementary Table X. Analytical scope of seven sensitivity analyses.",
    "sensitivity_new_si_table2_selected_coefficient_robustness": "Supplementary Table X. Outcome-tail and denominator-restriction sensitivity for selected model terms.",
    "sensitivity_new_si_table3_covariance_interval_robustness": "Supplementary Table X. Covariance-estimator sensitivity for selected baseline-supported terms.",
}

table_notes = {
    "sensitivity_new_si_table1_analysis_scope": "Figure-based analyses refer to the five graphical sensitivity checks already shown in this section; table-based analyses refer to the two additional tabular checks generated below. WLS = weighted least squares; OLS = ordinary least squares; CI = confidence interval.",
    "sensitivity_new_si_table2_selected_coefficient_robustness": "Values are percent changes in the log1p-transformed conversion ratio, with 95% confidence intervals in parentheses. Infection-to-incidence uses sl_pfinc/sl_pfinf; incidence-to-mortality uses sl_pfmort/sl_pfinc. Denominator means the denominator of the corresponding conversion ratio and is also used as the WLS weight. EDI = Environmental Deprivation Index; same direction compares each restricted-sample estimate with the full-sample estimate; CI excludes zero counts how many of the five sample definitions have 95% CIs that do not cross zero.",
    "sensitivity_new_si_table3_covariance_interval_robustness": "Values compare uncertainty estimates for the same point estimate under country-clustered, HC3 robust, HC1 robust and classical WLS standard errors. All intervals exclude zero indicates whether all four 95% confidence intervals support the same non-zero direction. EDI = Environmental Deprivation Index; WLS = weighted least squares; CI = confidence interval.",
}
outputs = {
    "sensitivity_new_si_table1_analysis_scope": table1,
    "sensitivity_new_si_table2_selected_coefficient_robustness": table2,
    "sensitivity_new_si_table3_covariance_interval_robustness": table3,
}
for stem, df in outputs.items():
    write_table_bundle(df, stem, captions[stem], table_notes.get(stem))

workbook = ADDITIONAL_TABLE_DIR / "sensitivity_new_si_manuscript_style_tables.xlsx"
try:
    with pd.ExcelWriter(workbook) as writer:
        table1.to_excel(writer, sheet_name="analysis_scope", index=False)
        table2.to_excel(writer, sheet_name="selected_coefficients", index=False)
        table3.to_excel(writer, sheet_name="covariance_intervals", index=False)
except Exception as exc:
    print(f"Skipped combined workbook export: {exc}")

print("Saved manuscript-style SI tables:")
for stem, df in outputs.items():
    print(f"- {stem}: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Output directory: {ADDITIONAL_TABLE_DIR}")

for stem, df in outputs.items():
    print("\n" + captions[stem])
    if table_notes.get(stem):
        print("Note. " + table_notes[stem])
    if "display" in globals():
        display(df)
    else:
        print(df.to_string(index=False))
